In [0]:
%sql
CREATE OR REPLACE TABLE mashup_learning.stocks.gold_daily_metrics
USING DELTA
AS
WITH staged AS (
  SELECT
    trade_date, previous_day_close, daily_return, rolling_avg_7d,
    open, high, low, close, volume, dividends, stock_splits, ticker,
    COUNT(volume) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS vol_count_20d,
    AVG(volume) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS raw_avg_volume_20d,
    COUNT(daily_return) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) as ret_count_20d,
    COUNT(*) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS row_count_20d
    -- other raw window calcs here
  FROM mashup_learning.stocks.silver_daily_prices
)
SELECT
  trade_date, previous_day_close, daily_return, rolling_avg_7d,
  open, high, low, close, volume, dividends, stock_splits, ticker,
  CURRENT_TIMESTAMP() AS gold_processed_at,
  CASE WHEN vol_count_20d < 20 THEN NULL ELSE raw_avg_volume_20d END AS rolling_avg_volume_20d,
  CASE WHEN vol_count_20d < 20 THEN NULL ELSE volume / raw_avg_volume_20d END AS relative_volume_20d,
  CASE WHEN ret_count_20d < 20 THEN NULL ELSE STDDEV(daily_return) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW )end as return_volatility_20d,
  CASE WHEN row_count_20d < 20 THEN NULL ELSE AVG(ABS(high - low) / close) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) END as swing_volatility_20d
FROM staged;